# Partial Differential Equations: The (Hyperbolic) Wave Equation

---
REFS:
- Landau, Páez, Bordeianu, *Computational Physics*, ch. on waves on a string
- Langtangen & Linge, [*Finite Difference Computing with PDEs*](https://hplgit.github.io/fdm-book/doc/pub/book/html/fdm-book.html), ch. 2
- [Wave equation (Wikipedia)](https://en.wikipedia.org/wiki/Wave_equation)
- Previous notebooks: [Relaxation](FiniteDifferences-Relaxation.ipynb), [Heat equation](PDE-Parabolic-Heat.ipynb), and [leap-frog in molecular dynamics](../08-MolecularDynamics/MolecularDynamics-MacroBodies.ipynb)
---

```{admonition} Learning goals
:class: tip
After this notebook you should be able to
1. discretize the wave equation with the leap-frog scheme, including its special first step;
2. derive and observe the CFL stability condition $r = c\,\Delta t/h \le 1$;
3. validate the solver with exact solutions (standing waves, d'Alembert) and conservation of energy;
4. implement fixed, free, periodic and absorbing boundaries and explain the reflections;
5. recognize numerical dispersion, and solve the wave equation in two dimensions.
```

In [ ]:
# Setup: all imports and small helpers used in the notebook
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, Image, display

sys.path.insert(0, os.path.abspath(os.path.join("..", "..", "utils")))
from embed import embed          # shared helper to embed videos and web pages

os.makedirs("fig", exist_ok=True)


def animate_1d(x, frames, times, ylabel="$u$", interval=60):
    """Animate a list of 1D profiles u(x) at the given times."""
    fig, ax = plt.subplots(figsize=(6, 3.5))
    line, = ax.plot(x, frames[0], lw=2)
    lim = 1.1*max(np.abs(f).max() for f in frames)
    ax.set(xlim=(x[0], x[-1]), ylim=(-lim, lim), xlabel="$x$", ylabel=ylabel)
    title = ax.set_title("")

    def update(k):
        line.set_ydata(frames[k])
        title.set_text(f"t = {times[k]:.3f}")
        return line, title

    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval)
    plt.close(fig)
    return HTML(ani.to_jshtml())


def animate_2d(frames, times, interval=80):
    """Animate a list of 2D fields u[i, j] (i along x) on the unit square."""
    fig, ax = plt.subplots(figsize=(4.8, 4))
    lim = max(np.abs(f).max() for f in frames[1:])
    im = ax.imshow(frames[0].T, origin="lower", extent=(0, 1, 0, 1), cmap="RdBu_r", vmin=-lim, vmax=lim)
    plt.colorbar(im, ax=ax, label="$u$")
    ax.set(xlabel="$x$", ylabel="$y$")
    title = ax.set_title("")

    def update(k):
        im.set_data(frames[k].T)
        title.set_text(f"t = {times[k]:.3f}")
        return im, title

    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=interval)
    plt.close(fig)
    return HTML(ani.to_jshtml())

## Vibrations and waves

$$
\frac{\partial^2 u}{\partial t^2} = c^2 \nabla^2 u
$$

Unlike the heat equation, which describes irreversible spreading, the wave equation describes **reversible propagation at a finite speed** $c$. It is the typical **hyperbolic** equation: we need the initial displacement *and* the initial velocity, plus boundary conditions.

| System | Field $u$ | Wave speed $c$ |
|---|---|---|
| Vibrating string | transverse displacement | $\sqrt{T/\mu}$ ($T$ = tension, $\mu$ = mass per length) |
| Sound | pressure or density | $\approx 343$ m/s in air |
| Light | $\vec E$, $\vec B$ fields | $1/\sqrt{\epsilon_0\mu_0} \approx 3\times10^8$ m/s |
| Seismic P-waves | rock displacement | $\sim 6$ km/s in the crust |

```{figure} https://upload.wikimedia.org/wikipedia/commons/1/1f/Wave_equation_1D_fixed_endpoints.gif
:width: 60%
:align: center
A pulse on a string with fixed ends (Wikimedia Commons).
```

In [ ]:
embed("https://visualpde.com/sim/?preset=waveEquation3D")

More demos: [VisualPDE 1D wave](https://visualpde.com/sim/?preset=waveEquation1D), [PhET wave on a string](https://phet.colorado.edu/sims/html/wave-on-a-string/latest/wave-on-a-string_all.html), [Falstad ripple tank](https://falstad.com/ripple/), [acoustics animations](https://www.acs.psu.edu/drussell/demos.html).

### Derivation: a vibrating string

```{figure} https://upload.wikimedia.org/wikipedia/commons/1/15/StringParameters.svg
:width: 40%
:align: center
```

Take a small piece of string between $x$ and $x + dx$, with mass $\mu\,dx$, under tension $T$. If the string makes a small angle $\theta(x) \approx \partial u/\partial x$ with the horizontal, the net vertical force is $T\left[\sin\theta(x + dx) - \sin\theta(x)\right] \approx T\,\frac{\partial^2 u}{\partial x^2}\,dx$. Newton's second law gives

$$
\mu\,dx\,\frac{\partial^2 u}{\partial t^2} = T\,\frac{\partial^2 u}{\partial x^2}\,dx
\quad\Longrightarrow\quad
\frac{\partial^2 u}{\partial t^2} = c^2\,\frac{\partial^2 u}{\partial x^2}, \qquad c = \sqrt{T/\mu}.
$$

### D'Alembert's solution

On an infinite line every solution has the form

$$
u(x, t) = f(x - ct) + g(x + ct),
$$

a shape $f$ moving to the right and a shape $g$ moving to the left, both at speed $c$ and **without changing shape**. With zero initial velocity, $f = g = u(x, 0)/2$: the initial pulse splits into two halves. Information travels at exactly speed $c$, which will be the key to stability.

## The leap-frog scheme

With $x_i = i h$, $t_n = n\,\Delta t$ and centered second differences in both time and space:

$$
\frac{u_i^{n+1} - 2u_i^n + u_i^{n-1}}{\Delta t^2} = c^2\,\frac{u_{i+1}^n - 2u_i^n + u_{i-1}^n}{h^2}
\quad\Longrightarrow\quad
\boxed{u_i^{n+1} = 2u_i^n - u_i^{n-1} + r^2\left(u_{i+1}^n - 2u_i^n + u_{i-1}^n\right), \qquad r = \frac{c\,\Delta t}{h}.}
$$

The scheme is explicit and second order in both $\Delta t$ and $h$. The number $r$ is the **Courant number**.

```{figure} fig/wave-discreteSol.png
:width: 80%
:align: center
The leap-frog stencil uses two old time levels to compute the new one.
```

```{admonition} You have seen this before: molecular dynamics
:class: note
Write the semi-discrete equation as $\ddot u_i = \frac{c^2}{h^2}\left(u_{i+1} - 2u_i + u_{i-1}\right)$. This is Newton's law for a **chain of masses connected by springs**, each mass pulled by its two neighbours. The update above is exactly the Verlet/leap-frog integrator of the [molecular dynamics notebook](../08-MolecularDynamics/MolecularDynamics-MacroBodies.ipynb), $x^{n+1} = 2x^n - x^{n-1} + a^n\Delta t^2$. A string is the continuum limit of that chain, and leap-frog inherits its good energy behaviour.
```

### The first step

Leap-frog needs two time levels, but at $t = 0$ we only know $u_i^0$ and the velocity $v_i = \partial_t u(x_i, 0)$. The centered velocity $v_i \approx (u_i^1 - u_i^{-1})/(2\Delta t)$ defines a "ghost" level $u_i^{-1} = u_i^1 - 2\Delta t\, v_i$. Inserting it into the leap-frog formula for $n = 0$ gives a second-order first step:

$$
u_i^1 = u_i^0 + \Delta t\, v_i + \frac{r^2}{2}\left(u_{i+1}^0 - 2u_i^0 + u_{i-1}^0\right).
$$

### The solver

The solver below takes the initial displacement and velocity, the Courant number and the number of steps. Boundary conditions are given as a function `bc(u_new, u, u_prev, r)` that fixes the end values of the new time level; by default the end values are kept fixed. It returns the profiles every `nsave` steps.

In [ ]:
def leapfrog_step(u, u_prev, r):
    """One leap-frog step for the interior points; the end values are copied from u."""
    u_new = u.copy()
    u_new[1:-1] = 2*u[1:-1] - u_prev[1:-1] + r**2*(u[2:] - 2*u[1:-1] + u[:-2])
    return u_new


def solve_wave_1d(u0, v0, r, dt, nsteps, bc=None, nsave=1):
    """Leap-frog solution of u_tt = c^2 u_xx.

    u0, v0 : initial displacement and velocity (arrays)
    r      : Courant number c*dt/h;  dt: time step;  nsteps: number of steps
    bc     : function bc(u_new, u, u_prev, r) that sets the end values of u_new (default: fixed ends)
    Returns an array with the profiles at steps 0, nsave, 2*nsave, ...
    """
    u_prev = u0.copy()
    u = u0.copy()
    u[1:-1] = u0[1:-1] + dt*v0[1:-1] + 0.5*r**2*(u0[2:] - 2*u0[1:-1] + u0[:-2])   # first step
    frames = [u0.copy()]
    for n in range(1, nsteps + 1):          # here u holds the solution at step n
        if n % nsave == 0:
            frames.append(u.copy())
        if n == nsteps:
            break
        u_new = leapfrog_step(u, u_prev, r)
        if bc is not None:
            bc(u_new, u, u_prev, r)
        u_prev, u = u, u_new
    return np.array(frames)

(wave-plucked)=
### First example: a plucked string

A Gaussian bump starts at rest in the middle of a string with fixed ends ($L = 1$, $c = 1$, $N = 201$, $r = 0.9$). As d'Alembert predicts, it splits into two half-height pulses, which reflect at the fixed ends **upside down** and meet again at $t = 1$ inverted, and recover the initial shape at $t = 2$.

In [ ]:
N, c, r = 201, 1.0, 0.9
x = np.linspace(0, 1, N)
h = x[1] - x[0]
dt = r*h/c
u0 = np.exp(-(x - 0.5)**2/(2*0.05**2))
u0[0] = u0[-1] = 0.0
v0 = np.zeros(N)
nsteps = int(round(2.0/dt))
frames = solve_wave_1d(u0, v0, r, dt, nsteps)
times = dt*np.arange(len(frames))

fig, axes = plt.subplots(2, 3, figsize=(12, 5.5), sharex=True, sharey=True)
for ax, t_show in zip(axes.flat, [0, 0.2, 0.4, 0.6, 1.0, 2.0]):
    k = int(round(t_show/dt))
    ax.plot(x, frames[k], lw=1.5)
    ax.set_title(f"t = {times[k]:.2f}")
    ax.axhline(0, color="gray", lw=0.5)
for ax in axes[1]:
    ax.set_xlabel("$x$")
for ax in axes[:, 0]:
    ax.set_ylabel("$u$")
fig.suptitle(f"Plucked string with fixed ends (r = {r})")
plt.tight_layout()
plt.show()

In [ ]:
animate_1d(x, frames[::12], times[::12])

## Stability: the CFL condition

### Von Neumann analysis

Insert a Fourier mode $u_i^n = G^n e^{\mathrm{i}kx_i}$, where $G = u^{n+1}/u^n$ is the amplification factor per step. With $s = \sin(kh/2)$, the leap-frog formula gives a quadratic equation for $G$:

$$
G^2 - 2\left(1 - 2r^2s^2\right)G + 1 = 0 .
$$

The product of its two roots is 1. If $r s \le 1$, the roots are complex conjugates with $|G| = 1$: the mode oscillates without growing or decaying, as a wave should. If $r s > 1$, the roots are real and one of them has $|G| > 1$: the mode grows exponentially. The worst mode is the saw-tooth ($s = 1$), so stability requires

$$
\boxed{r = \frac{c\,\Delta t}{h} \le 1} \qquad \text{(Courant–Friedrichs–Lewy condition).}
$$

In $d$ dimensions with equal spacings, the condition is $r \le 1/\sqrt d$.

**Physical meaning.** In one step, the numerical scheme can only move information by one cell, i.e. at speed $h/\Delta t$. The CFL condition says that this numerical speed must be at least the physical speed $c$: the numerical domain of dependence must contain the physical one.

Notice the difference with the heat equation: here $\Delta t \propto h$ (not $h^2$), which is much less restrictive.

### Seeing the instability

In [ ]:
N = 101
x = np.linspace(0, 1, N)
h = x[1] - x[0]
fig, ax = plt.subplots(figsize=(6, 4))
for r in [1.0, 1.02]:
    dt = r*h
    frames = solve_wave_1d(np.sin(np.pi*x), np.zeros(N), r, dt, 300)
    ax.semilogy(np.abs(frames).max(axis=1), label=f"r = {r}")
ax.set(xlabel="step", ylabel="max $|u|$", title="Leap-frog at and just above the CFL limit")
ax.legend()
plt.show()

```{exercise} Reading the CFL instability
:label: ex-wave-cfl
1. Solve the quadratic equation for $G$ with $s = 1$ and $r = 1.02$. What is the largest $|G|$? Is it consistent with the slope of the plot above?
2. About how many steps does it take for round-off errors ($\sim10^{-16}$) to grow to size 1? Compare with the plot.
3. Repeat item 1 for $r = 1.05$.

```

## Validation

### Standing waves: an exact solution

With fixed ends, $u(x, 0) = \sin(\pi x)$ and zero velocity, the exact solution is the standing wave $u = \sin(\pi x)\cos(\pi c t)$, with period $T = 2L/c = 2$. We measure the error at a quarter period, $t = 0.5$, for several Courant numbers, and the order of accuracy with $r = 0.5$ fixed. (Measuring after exactly one full period would be misleading: there the error of the phase cancels to first order, and the error looks fourth order.)

In [ ]:
c = 1.0
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

N = 51
x = np.linspace(0, 1, N)
h = x[1] - x[0]
rs = [0.3, 0.5, 0.7, 0.9, 1.0]
errs_r = []
for r in rs:
    dt = r*h/c
    nsteps = int(round(0.5/dt))
    frames = solve_wave_1d(np.sin(np.pi*x), np.zeros(N), r, dt, nsteps)
    errs_r.append(np.max(np.abs(frames[-1] - np.sin(np.pi*x)*np.cos(np.pi*c*nsteps*dt))) + 1e-17)
ax1.semilogy(rs, errs_r, "o-")
ax1.set(xlabel="Courant number $r$", ylabel="max error at $t = 0.5$", title=f"N = {N}")

Ns = np.array([21, 41, 81, 161, 321])
errs_h = []
for N in Ns:
    x = np.linspace(0, 1, N)
    h = x[1] - x[0]
    dt = 0.5*h/c
    nsteps = int(round(0.5/dt))
    frames = solve_wave_1d(np.sin(np.pi*x), np.zeros(N), 0.5, dt, nsteps, nsave=nsteps)
    errs_h.append(np.max(np.abs(frames[-1] - np.sin(np.pi*x)*np.cos(np.pi*c*nsteps*dt))))
hs = 1/(Ns - 1)
slope = np.polyfit(np.log(hs), np.log(errs_h), 1)[0]
ax2.loglog(hs, errs_h, "o-", label=f"r = 0.5 (slope {slope:.2f})")
ax2.loglog(hs, errs_h[0]*(hs/hs[0])**2, "k--", lw=1, label=r"$\propto h^2$")
ax2.set_xticks(hs, [f"1/{N-1}" for N in Ns])
ax2.minorticks_off()
ax2.set(xlabel="$h$", ylabel="max error at $t = 0.5$", title="Order of accuracy")
ax2.legend()
plt.tight_layout()
plt.show()

The error drops to round-off level at $r = 1$. In 1D and only at $r = 1$, leap-frog reproduces d'Alembert's solution *exactly*: each value is copied one cell per step, which is precisely the exact motion. For $r < 1$ the scheme is second order.

### Numerical dispersion

The exact equation is non-dispersive: all wavelengths travel at speed $c$. The scheme is not. Inserting $u_i^n = e^{\mathrm{i}(kx_i - \omega t_n)}$ gives the numerical dispersion relation

$$
\omega_\text{num} = \frac{2}{\Delta t}\arcsin\!\left(r\,\sin\frac{kh}{2}\right),
$$

which tends to $\omega = ck$ for long waves ($kh \ll 1$) but makes short waves travel **slower** than $c$, unless $r = 1$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
kh = np.linspace(1e-6, np.pi, 300)
for r in [0.3, 0.7, 0.9, 1.0]:
    omega_dt = 2*np.arcsin(r*np.sin(kh/2))
    ax1.plot(kh/np.pi, (omega_dt/kh)/r, label=f"r = {r}")      # numerical phase speed / c
ax1.set(xlabel=r"$kh/\pi$", ylabel=r"$c_\mathrm{num}/c$", title="Numerical phase speed (leap-frog)")
ax1.legend()

# A very narrow pulse travelling to the right: u = f(x - ct), so v0 = -c f'(x)
N = 401
x = np.linspace(0, 1, N)
h = x[1] - x[0]
sigma, x0, t_end = 0.008, 0.2, 0.6
f = lambda s: np.exp(-(s - x0)**2/(2*sigma**2))
for r in [0.5, 1.0]:
    dt = r*h
    nsteps = int(round(t_end/dt))
    v0 = (x - x0)/sigma**2*f(x)                      # -c f'(x) with c = 1
    frames = solve_wave_1d(f(x), v0, r, dt, nsteps, nsave=nsteps)
    ax2.plot(x, frames[-1], label=f"r = {r}")
ax2.plot(x, f(x - t_end), "k--", lw=1, label="exact")
ax2.set(xlim=(0.65, 0.9), xlabel="$x$", ylabel="$u$", title=f"Narrow pulse after t = {t_end}")
ax2.legend()
plt.tight_layout()
plt.show()

With $r = 0.5$ the short-wave components of the narrow pulse lag behind and form a trail of ripples behind the pulse. Resolving the pulse with more points per width ($\sigma/h \gtrsim 10$) removes the problem.

### Energy

The energy of the string, $E = \int \left[\tfrac12 u_t^2 + \tfrac12 c^2 u_x^2\right]dx$, is conserved by the exact equation. Leap-frog does not dissipate energy (unlike the heat equation, where decay is the physics): a discrete version of $E$ oscillates slightly but does not drift, just like the energy of a Verlet integration in molecular dynamics. You will check it in an exercise.

## Boundary conditions and reflections

| Boundary | Physical example | Condition | Reflection |
|---|---|---|---|
| Fixed (Dirichlet) | string clamped to a wall | $u = 0$ | inverted |
| Free (Neumann) | string end on a frictionless ring, open pipe | $u_x = 0$ | upright |
| Periodic | ring, or a crystal | $u(0) = u(L)$ | none: the wave re-enters on the other side |
| Absorbing | infinite string, open space | outgoing wave only | (ideally) none |

```{figure} fig/wave-bc.png
:width: 90%
:align: center
```

For the boundary demos we send a single pulse to the right. For a right-moving wave $u = f(x - ct)$, the initial velocity is $v_0 = -c f'(x)$. With fixed ends (the default), the reflected pulse comes back inverted. At $t = 0.7$ the pulse sits on the wall: think of the reflection as an "image" pulse coming from outside, inverted for a fixed end (so the two cancel and $u = 0$ at the wall) and upright for a free end (so they add up to height 2).

In [ ]:
N, c, r = 201, 1.0, 0.9
x = np.linspace(0, 1, N)
h = x[1] - x[0]
dt = r*h/c
x0, sigma = 0.3, 0.04
u0_right = np.exp(-(x - x0)**2/(2*sigma**2))
v0_right = c*(x - x0)/sigma**2*u0_right              # -c f'(x)


def plot_reflection(frames, title):
    """Snapshots of a right-moving pulse before and after it reaches the right end."""
    fig, ax = plt.subplots(figsize=(7, 3.5))
    for t_show in [0.0, 0.4, 0.7, 1.0]:
        k = int(round(t_show/dt))
        ax.plot(x, frames[k], label=f"t = {t_show}")
    ax.set(xlabel="$x$", ylabel="$u$", title=title, ylim=(-2.1, 2.1))
    ax.legend(fontsize=8, loc="lower left")
    plt.tight_layout()
    plt.show()


frames = solve_wave_1d(u0_right, v0_right, r, dt, int(round(1.0/dt)))
plot_reflection(frames, "Fixed end: inverted reflection")

```{exercise} Free and periodic boundaries
:label: ex-wave-free-periodic
1. **Free ends.** The ghost point $u_{-1} = u_1$ (zero slope) turns the leap-frog formula at $i = 0$ into $u_0^{n+1} = 2u_0^n - u_0^{n-1} + 2r^2\left(u_1^n - u_0^n\right)$, and similarly at the right end. Complete `bc_free(u_new, u, u_prev, r)` (two lines).
2. **Periodic.** Treat $x = 0$ and $x = 1$ as the same point: the left neighbour of $u_0$ is $u_{N-2}$. Complete `bc_periodic` by computing `u_new[0]` with the leap-frog formula using `u[1]` and `u[-2]` as neighbours, and set `u_new[-1] = u_new[0]`.
3. Run the cell after the functions. Does the pulse reflect upright at a free end? Does it re-enter on the left with periodic ends?
```

In [ ]:
def bc_free(u_new, u, u_prev, r):
    """Free ends: zero slope (ghost points u[-1] = u[1])."""
    # YOUR CODE HERE
    pass


def bc_periodic(u_new, u, u_prev, r):
    """Periodic ends: x = 0 and x = L are the same point."""
    # YOUR CODE HERE
    pass

In [ ]:
# Given: run it after completing bc_free and bc_periodic
frames = solve_wave_1d(u0_right, v0_right, r, dt, int(round(1.0/dt)), bc=bc_free)
plot_reflection(frames, "Free end: upright reflection")
frames = solve_wave_1d(u0_right, v0_right, r, dt, int(round(1.0/dt)), bc=bc_periodic)
plot_reflection(frames, "Periodic: the pulse re-enters on the left")

```{exercise} Absorbing boundaries
:label: ex-wave-absorbing
To simulate an infinite string we want the outgoing wave to leave without reflection. At the right end the wave must satisfy the one-way equation $u_t + c\,u_x = 0$. Discretizing it at the boundary gives the first-order **Mur condition**

$$
u_{N-1}^{n+1} = u_{N-2}^{n} + \frac{r - 1}{r + 1}\left(u_{N-2}^{n+1} - u_{N-1}^{n}\right),
$$

and, at the left end, $u_0^{n+1} = u_1^n + \frac{r-1}{r+1}\left(u_1^{n+1} - u_0^n\right)$. (The naive copy `u_new[-1] = u[-2]` is the special case $r = 1$.)
1. Complete `bc_absorbing` (two lines; `u_new[1:-1]` already holds the new interior values).
2. Run the next cell. It sends the right-moving pulse with $r = 0.9$ and prints the amplitude $\max|u|$ left at $t = 1$ for fixed ends, the naive copy and the Mur condition. Why does the $r$-dependent factor matter?
```

In [ ]:
def bc_absorbing(u_new, u, u_prev, r):
    """First-order Mur absorbing boundaries at both ends."""
    # YOUR CODE HERE
    pass

In [ ]:
# Given: run it after completing bc_absorbing
def bc_naive(u_new, u, u_prev, r):
    u_new[0] = u[1]
    u_new[-1] = u[-2]

nsteps = int(round(1.0/dt))
for name, bc_f in [("fixed", None), ("naive copy", bc_naive), ("Mur", bc_absorbing)]:
    frames = solve_wave_1d(u0_right, v0_right, r, dt, nsteps, bc=bc_f)
    print(f"{name:10s}: max |u| left at t = 1: {np.max(np.abs(frames[-1])):.2e}")
plot_reflection(frames, "Mur absorbing boundary (r = 0.9)")

## Two dimensions

For a membrane, $u_{tt} = c^2(u_{xx} + u_{yy})$. With $h_x = h_y = h$, leap-frog uses the five-point stencil again:

$$
u_{i,j}^{n+1} = 2u_{i,j}^n - u_{i,j}^{n-1} + r^2\left(u_{i+1,j}^n + u_{i-1,j}^n + u_{i,j+1}^n + u_{i,j-1}^n - 4u_{i,j}^n\right), \qquad r \le \frac{1}{\sqrt2}.
$$

```{figure} fig/wave-2d-points.png
:width: 80%
:align: center
```

In [ ]:
def laplacian5(u):
    """Five-point sum u[i+1,j] + u[i-1,j] + u[i,j+1] + u[i,j-1] - 4u[i,j] on the interior."""
    return u[2:, 1:-1] + u[:-2, 1:-1] + u[1:-1, 2:] + u[1:-1, :-2] - 4*u[1:-1, 1:-1]


def solve_wave_2d(u0, r, nsteps, nsave=1):
    """Leap-frog for the 2D wave equation: zero initial velocity, u = 0 on the edges."""
    u_prev = u0.copy()
    u = u0.copy()
    u[1:-1, 1:-1] = u0[1:-1, 1:-1] + 0.5*r**2*laplacian5(u0)       # first step (v0 = 0)
    frames = [u0.copy()]
    for n in range(1, nsteps + 1):
        if n % nsave == 0:
            frames.append(u.copy())
        if n == nsteps:
            break
        u_new = np.zeros_like(u)
        u_new[1:-1, 1:-1] = 2*u[1:-1, 1:-1] - u_prev[1:-1, 1:-1] + r**2*laplacian5(u)
        u_prev, u = u, u_new
    return np.array(frames)


N, c, r = 101, 1.0, 0.5
x = np.linspace(0, 1, N)
h = x[1] - x[0]
dt = r*h/c
XX, YY = np.meshgrid(x, x, indexing="ij")
u0 = np.exp(-((XX - 0.4)**2 + (YY - 0.5)**2)/(2*0.04**2))
u0[0, :] = u0[-1, :] = u0[:, 0] = u0[:, -1] = 0.0
frames = solve_wave_2d(u0, r, 320, nsave=8)
animate_2d(frames, dt*8*np.arange(len(frames)))

Notice two differences with 1D: the circular wave **decreases in amplitude** as it spreads (the energy is spread over a growing circle, so $u \sim 1/\sqrt{\rho}$), and it leaves a **trailing wake** behind the front. In 2D the sharp-front property of d'Alembert's solution is lost (it holds again in 3D: Huygens' principle).

### Normal modes of a square drum

A membrane with fixed edges on $[0, L]^2$ has the normal modes

$$
u_{mn}(x, y, t) = \sin\frac{m\pi x}{L}\,\sin\frac{n\pi y}{L}\,\cos(\omega_{mn}t), \qquad \omega_{mn} = \frac{\pi c}{L}\sqrt{m^2 + n^2}.
$$

Unlike a string, the frequencies are *not* multiples of a fundamental (which is why a drum sounds less "musical" than a guitar), and different modes can have the same frequency: $(1,2)$ and $(2,1)$ are **degenerate**, like states in quantum mechanics. See [the modes of a drum](https://www.youtube.com/watch?v=7nG4Y7FU-dQ).

## More exercises

```{exercise} Splitting pulse and wave speed
:label: ex-wave-speed
A Gaussian $u(x, 0) = e^{-x^2/(2\cdot0.2^2)}$ at rest on a long string $x \in [-5, 5]$ ($N = 1001$, $c = 1$, $r = 0.9$) splits into two pulses. We follow it up to $t = 3$, before the pulses reach the ends.
1. Complete `peak_position(x, u)`, which returns the position of the maximum of `u` among the points with $x > 0$ (use a mask and `np.argmax`).
2. Run the next cell, which tracks the right-moving peak every 10 steps and fits a straight line. Compare the slope with $c$. Is the height of each pulse $1/2$?
```

In [ ]:
def peak_position(x, u):
    """x position of the maximum of u, looking only at x > 0."""
    # YOUR CODE HERE
    pass

In [ ]:
# Given: run it after completing peak_position
N, c, r = 1001, 1.0, 0.9
x = np.linspace(-5, 5, N)
h = x[1] - x[0]
dt = r*h/c
u0 = np.exp(-x**2/(2*0.2**2))
frames = solve_wave_1d(u0, np.zeros(N), r, dt, int(3.0/dt), nsave=10)
times = 10*dt*np.arange(len(frames))
pos = np.array([peak_position(x, f) for f in frames])
mask = times > 0.5                           # once the two halves are separated
speed, x_start = np.polyfit(times[mask], pos[mask], 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(times, pos, "o", ms=3, label="peak position")
ax1.plot(times[mask], x_start + speed*times[mask], "k-", lw=1, label=f"fit: speed = {speed:.4f} (c = {c})")
ax1.set(xlabel="$t$", ylabel="peak position", title="Right-moving half")
ax1.legend()
ax2.plot(x, frames[0], "k:", label="t = 0")
ax2.plot(x, frames[-1], label=f"t = {times[-1]:.2f}")
ax2.axhline(0.5, color="gray", ls="--", lw=1)
ax2.set(xlabel="$x$", ylabel="$u$", title="Two half-height pulses")
ax2.legend()
plt.tight_layout()
plt.show()

```{exercise} Standing waves and their frequencies
:label: ex-wave-modes
Start from $u(x, 0) = \sin(n\pi x)$ with zero velocity and fixed ends ($L = 1$, $c = 1$, $N = 101$, $r = 0.9$), simulate up to $t = 20$ and record $u$ at $x = 1/4$.
1. Complete `spectrum(signal, dt)`, which returns the frequencies `np.fft.rfftfreq(len(signal), dt)` and the amplitudes `np.abs(np.fft.rfft(signal))`.
2. Run the next cell for $n = 1, 2, 3$ and for the superposition of $n = 1$ and $n = 2$. Are the peaks at $f_n = nc/(2L)$? Which peaks appear for the superposition?
```

In [ ]:
def spectrum(signal, dt):
    """Frequencies and amplitude spectrum of a signal sampled every dt."""
    # YOUR CODE HERE
    pass

In [ ]:
# Given: run it after completing spectrum
N, c, r = 101, 1.0, 0.9
x = np.linspace(0, 1, N)
h = x[1] - x[0]
dt = r*h/c
iq = N//4                                     # x = 1/4
fig, ax = plt.subplots(figsize=(7, 4))
cases = [("n = 1", np.sin(np.pi*x)), ("n = 2", np.sin(2*np.pi*x)),
         ("n = 3", np.sin(3*np.pi*x)), ("n = 1 + 2", np.sin(np.pi*x) + np.sin(2*np.pi*x))]
for label, u0 in cases:
    signal = solve_wave_1d(u0, np.zeros(N), r, dt, int(20/dt))[:, iq]
    freqs, amp = spectrum(signal, dt)
    ax.plot(freqs, amp/amp.max(), label=label)
for n in [1, 2, 3]:
    ax.axvline(n*c/2, color="gray", ls=":", lw=1)
ax.set(xlim=(0, 2.5), xlabel="frequency $f$", ylabel="normalized spectrum", title="Spectrum of u(1/4, t); dotted: $f_n = n/2$")
ax.legend()
plt.show()

```{exercise} Energy conservation
:label: ex-wave-energy
For the plucked string of {ref}`wave-plucked` (fixed ends, $N = 201$), the discrete energy at step $n$ is

$$
E^n = h\sum_i \left[\frac12\left(\frac{u_i^{n+1} - u_i^{n-1}}{2\Delta t}\right)^2 + \frac{c^2}{2}\left(\frac{u_{i+1}^n - u_i^n}{h}\right)^2\right].
$$

1. Complete `wave_energy(U, h, dt, c)`, where `U[n]` is the profile at step $n$. The velocity `ut` and slope `ux` at steps $1, \dots, n-1$ are already computed; write the line for `E` (sum along `axis=1`).
2. Run the next cell for $r = 0.5$, $0.9$ and $1.0$ over 2000 steps. Does the energy drift, or only oscillate? Compare with the heat equation, where the "energy" $\int u^2\,dx$ always decreases.
```

In [ ]:
def wave_energy(U, h, dt, c):
    """Discrete energy at the steps 1 .. len(U)-2 of the frames U (fixed ends)."""
    ut = (U[2:] - U[:-2])/(2*dt)              # velocity at steps 1 .. n-1
    ux = np.diff(U[1:-1], axis=1)/h           # slope at the same steps
    # YOUR CODE HERE
    pass
    return E

In [ ]:
# Given: run it after completing wave_energy
N, c = 201, 1.0
x = np.linspace(0, 1, N)
h = x[1] - x[0]
u0 = np.exp(-(x - 0.5)**2/(2*0.05**2))
u0[0] = u0[-1] = 0.0
fig, ax = plt.subplots(figsize=(7, 4))
for r in [0.5, 0.9, 1.0]:
    dt = r*h/c
    E = wave_energy(solve_wave_1d(u0, np.zeros(N), r, dt, 2000), h, dt, c)
    ax.plot(np.arange(1, len(E) + 1), (E - E[0])/E[0], lw=0.8, label=f"r = {r}")
ax.set(xlabel="step", ylabel="$(E^n - E^1)/E^1$", title="Relative change of the discrete energy")
ax.legend()
plt.show()

```{exercise} The square drum
:label: ex-wave-drum
We check the drum frequencies $\omega_{mn} = \pi\sqrt{m^2 + n^2}$ ($L = c = 1$) with `solve_wave_2d` ($N = 41$, $r = 0.5$, up to $t = 6$), starting from a single mode `u0 = np.sin(m*np.pi*XX)*np.sin(n*np.pi*YY)` and recording $u$ at $(1/4, 1/4)$.
1. Complete `period(t, signal)`: find the indices where the signal changes sign with `np.where(np.diff(np.sign(signal)) != 0)[0]`; there are two sign changes per period, so return twice the mean spacing of those times.
2. Run the next cell for the modes $(1, 1)$, $(1, 2)$ and $(2, 1)$. Compare with $T_{11} = 2/\sqrt2 \approx 1.414$ and $T_{12} = 2/\sqrt5 \approx 0.894$. Are $(1, 2)$ and $(2, 1)$ degenerate?
```

In [ ]:
def period(t, signal):
    """Oscillation period from the times where the signal changes sign."""
    # YOUR CODE HERE
    pass

In [ ]:
# Given: run it after completing period
N, r = 41, 0.5
x = np.linspace(0, 1, N)
h = x[1] - x[0]
dt = r*h
XX, YY = np.meshgrid(x, x, indexing="ij")
nsteps = int(6.0/dt)
iq = N//4
t = dt*np.arange(nsteps + 1)
fig, ax = plt.subplots(figsize=(8, 4))
for m, n in [(1, 1), (1, 2), (2, 1)]:
    u0 = np.sin(m*np.pi*XX)*np.sin(n*np.pi*YY)
    signal = solve_wave_2d(u0, r, nsteps)[:, iq, iq]
    T_exact = 2/np.sqrt(m**2 + n**2)
    ax.plot(t, signal, label=f"({m},{n}): T = {period(t, signal):.4f} (exact {T_exact:.4f})")
ax.set(xlabel="$t$", ylabel="$u(1/4, 1/4, t)$", title="Square drum modes")
ax.legend(fontsize=8, loc="lower left")
plt.show()

## Summary

| Topic | Key result |
|---|---|
| Scheme | leap-frog: $u^{n+1} = 2u^n - u^{n-1} + r^2\,\delta_x^2 u^n$, second order; special first step from $v_0$ |
| Stability | CFL: $r = c\Delta t/h \le 1$ (1D), $\le 1/\sqrt d$ in $d$ dimensions; $\Delta t \propto h$ |
| Accuracy | $O(h^2)$; exact in 1D at $r = 1$; short waves travel too slowly (numerical dispersion) |
| Energy | not dissipated: bounded oscillation, as in Verlet/MD |
| Boundaries | fixed: inverted reflection; free: upright; periodic: re-entry; Mur: absorbing |
| 2D | amplitude decays as the wave spreads; drum modes $\omega_{mn} \propto \sqrt{m^2 + n^2}$, with degeneracies |

Comparison of the three model equations of this chapter:

| | Laplace (elliptic) | Heat (parabolic) | Wave (hyperbolic) |
|---|---|---|---|
| Needs | boundary values | initial $u$ + boundaries | initial $u$ and $u_t$ + boundaries |
| Typical method | relaxation, sparse solve | FTCS, implicit θ-method | leap-frog |
| Explicit stability | — | $\Delta t \le h^2/(2\alpha)$ | $\Delta t \le h/c$ |
| Behaviour | smooth, no time | smoothing, irreversible | propagation, reversible |

## Further exercises
1. **Inhomogeneous string.** Use a position-dependent speed, $r_i = c(x_i)\Delta t/h$ in the update, with $c = 1$ for $x < 0.5$ and $c = 0.5$ for $x > 0.5$. Send a pulse and measure the reflected and transmitted amplitudes; compare with $R = (c_2 - c_1)/(c_2 + c_1)$. Which $c$ sets the CFL limit?
2. **Damped string.** Add a friction term, $u_{tt} = c^2u_{xx} - \gamma u_t$, and discretize $u_t$ with a centered difference. How does the energy decay?
3. **Plucked guitar string.** Start from a triangular shape (a string pulled at $x = 0.2$) and compute the spectrum. Compare the harmonic amplitudes with the Fourier series of the triangle.
4. **Absorbing walls in 2D.** Extend the Mur condition to the four edges of the membrane and simulate a pulse in "open space".
5. **Double slit.** In 2D, put a wall with two small openings (fixed $u = 0$ points) and a plane wave coming from the left. Observe the interference pattern (compare with the [ripple tank](https://falstad.com/ripple/)).